# 📖 Notebook 2: The Cache-Aside Pattern

Cache-aside (also called **lazy loading**) is the most common caching pattern and the one you should default to in interviews.

## How It Works

```
  Application               Cache (Redis)             Database (Postgres)
      │                         │                           │
      │── 1. Check cache ──────▶│                           │
      │◀── 2a. Cache HIT ──────│                           │
      │        (return data)    │                           │
      │                         │                           │
      │── 1. Check cache ──────▶│                           │
      │◀── 2b. Cache MISS ─────│                           │
      │── 3. Query database ───────────────────────────────▶│
      │◀── 4. Return data ─────────────────────────────────│
      │── 5. Store in cache ───▶│                           │
      │                         │                           │
```

The application is in control: it checks the cache, falls back to the database on a miss, and populates the cache for next time.

## Learning Objectives

- Implement cache-aside from scratch
- Measure cache hit/miss rates
- Understand when data gets stale
- See how cache-aside performs under load

In [ ]:
import psycopg2
import redis
import json
import time

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "caching_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

r = get_redis()
r.flushdb()  # start fresh
print("✅ Connected and Redis cleared")

## 🔧 Implementing Cache-Aside

Let's build the pattern step by step. We'll cache product data from our e-commerce database.

In [ ]:
# Tracking metrics so we can see hit/miss rates
stats = {"hits": 0, "misses": 0}

def get_product(product_id: int) -> dict:
    """
    Cache-Aside pattern:
    1. Check Redis for the product
    2. If found (HIT) → return it
    3. If not found (MISS) → query Postgres, store in Redis, return it
    """
    cache_key = f"product:{product_id}"
    
    # Step 1: Check the cache
    cached = r.get(cache_key)
    
    if cached:
        # Step 2a: Cache HIT — data is already in Redis
        stats["hits"] += 1
        return json.loads(cached)
    
    # Step 2b: Cache MISS — need to go to the database
    stats["misses"] += 1
    
    # Step 3: Query the database
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT p.id, p.name, p.price, p.rating_avg, p.view_count,
               p.stock_quantity, c.name as category
        FROM products p
        JOIN categories c ON p.category_id = c.id
        WHERE p.id = %s
    """, (product_id,))
    row = cursor.fetchone()
    conn.close()
    
    if not row:
        return None
    
    product = {
        "id": row[0], "name": row[1], "price": float(row[2]),
        "rating": float(row[3]), "views": row[4],
        "stock": row[5], "category": row[6]
    }
    
    # Step 5: Store in cache for next time
    r.set(cache_key, json.dumps(product))
    
    return product

print("✅ get_product() function ready")
print("   This implements the full cache-aside pattern.")

In [ ]:
# Let's call it a few times and watch the hit/miss behavior

stats = {"hits": 0, "misses": 0}

print("🔍 Calling get_product(42) three times:")
print()

for i in range(3):
    start = time.time()
    product = get_product(42)
    elapsed = (time.time() - start) * 1000
    
    call_num = i + 1
    was_hit = stats["hits"] > i  # did hits increase?
    status = "HIT ⚡" if i > 0 else "MISS 🐌"
    
    print(f"  Call {call_num}: {status}  {elapsed:.2f} ms")

print()
print(f"📊 Stats: {stats['hits']} hits, {stats['misses']} misses")
print(f"   Hit rate: {stats['hits']/(stats['hits']+stats['misses'])*100:.0f}%")
print()
print(f"📦 Product: {product['name']} — ${product['price']} ({product['category']})")
print()
print("💡 The first call is a MISS (goes to DB). Calls 2 and 3 are HITs (from Redis).")
print("   This is the 'lazy' in lazy loading — we only cache when someone asks for it.")

## 📈 Cache Hit Rate Under Realistic Traffic

In real systems, some products are viewed far more often than others (think: trending items).  
Let's simulate realistic traffic with a **Zipf distribution** — a few products get most of the views.

In [ ]:
import random

# Reset
r.flushdb()
stats = {"hits": 0, "misses": 0}

# Simulate 1000 requests with skewed distribution
# Most requests go to a small set of "popular" products
popular_products = list(range(1, 21))     # top 20 products (10% of catalog)
other_products = list(range(21, 201))      # remaining 180 products

requests = []
for _ in range(1000):
    # 80% of traffic goes to 10% of products (realistic power law)
    if random.random() < 0.8:
        requests.append(random.choice(popular_products))
    else:
        requests.append(random.choice(other_products))

# Process all requests using cache-aside
start = time.time()
for pid in requests:
    get_product(pid)
total_time = time.time() - start

total = stats["hits"] + stats["misses"]
hit_rate = stats["hits"] / total * 100

print("📊 Cache-Aside with Realistic Traffic (1000 requests)")
print("=" * 55)
print(f"   Cache hits:   {stats['hits']:>5}")
print(f"   Cache misses: {stats['misses']:>5}")
print(f"   Hit rate:     {hit_rate:.1f}%")
print(f"   Total time:   {total_time:.2f}s")
print(f"   Unique products cached: {len(r.keys('product:*'))}")
print()
print("💡 After the first round of misses, nearly all reads are cache hits!")
print("   The 80/20 rule means popular products stay warm in cache.")

## ⚠️ The Staleness Problem

Cache-aside has a downside: when data changes in the database, the cache doesn't know.  
Let's see this in action.

In [ ]:
# First, read a product (this caches it)
product = get_product(42)
print(f"1️⃣ Cached product price: ${product['price']}")

# Now update the price directly in the database
new_price = 99.99
conn = get_db()
cursor = conn.cursor()
cursor.execute("UPDATE products SET price = %s WHERE id = 42", (new_price,))
conn.commit()
conn.close()
print(f"2️⃣ Updated price in database to: ${new_price}")

# Read again — this will return the STALE cached value!
product_stale = get_product(42)
print(f"3️⃣ get_product(42) returns: ${product_stale['price']}")
print()

if product_stale['price'] != new_price:
    print("⚠️  STALE DATA! The cache still has the old price.")
    print("   The database says $99.99, but the cache says $" + str(product_stale['price']))
    print()
    print("💡 This is the #1 problem with cache-aside:")
    print("   The cache doesn't know when the database changes.")
    print("   We need cache INVALIDATION — covered in Notebook 4.")

In [ ]:
# Quick fix: invalidate (delete) the cache key after updating the database

def update_product_price(product_id: int, new_price: float):
    """Update price in DB and invalidate the cache."""
    # Step 1: Update the database
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("UPDATE products SET price = %s WHERE id = %s", (new_price, product_id))
    conn.commit()
    conn.close()
    
    # Step 2: Delete the stale cache entry
    r.delete(f"product:{product_id}")

# Now update with invalidation
update_product_price(42, 79.99)
print("1️⃣ Updated price to $79.99 and invalidated cache")

# Next read will be a MISS → fetch fresh data from DB → re-cache
product_fresh = get_product(42)
print(f"2️⃣ get_product(42) returns: ${product_fresh['price']}")
print()
print("✅ Fresh data! Invalidation ensures the next read gets the latest value.")

## 🏗️ Cache-Aside with Hash (Structured Data)

Instead of storing products as JSON strings, we can use Redis **Hashes** — they let us read/update individual fields without fetching the whole object.

In [ ]:
def get_product_hash(product_id: int) -> dict:
    """Cache-aside using Redis Hash instead of JSON string."""
    cache_key = f"product_hash:{product_id}"
    
    # Check cache — hgetall returns {} if key doesn't exist
    cached = r.hgetall(cache_key)
    if cached:
        # Redis returns all values as strings, so convert numeric fields
        cached["price"] = float(cached["price"])
        cached["rating"] = float(cached["rating"])
        cached["views"] = int(cached["views"])
        cached["stock"] = int(cached["stock"])
        return cached
    
    # Cache miss — fetch from DB
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT p.id, p.name, p.price, p.rating_avg, p.view_count,
               p.stock_quantity, c.name
        FROM products p JOIN categories c ON p.category_id = c.id
        WHERE p.id = %s
    """, (product_id,))
    row = cursor.fetchone()
    conn.close()
    
    if not row:
        return None
    
    product = {
        "id": str(row[0]), "name": row[1], "price": str(row[2]),
        "rating": str(row[3]), "views": str(row[4]),
        "stock": str(row[5]), "category": row[6]
    }
    
    # Store as hash — each field is stored separately
    r.hset(cache_key, mapping=product)
    
    # Convert back for return
    product["price"] = float(product["price"])
    product["rating"] = float(product["rating"])
    product["views"] = int(product["views"])
    product["stock"] = int(product["stock"])
    return product

# Demo: fetch a product
product = get_product_hash(10)
print(f"📦 Product: {product['name']}")
print(f"   Price: ${product['price']}, Stock: {product['stock']}")
print()

# With hashes, we can read just one field:
just_price = r.hget("product_hash:10", "price")
print(f"💰 Just the price (single field read): ${just_price}")

# Or update just one field:
r.hset("product_hash:10", "stock", "999")
updated_stock = r.hget("product_hash:10", "stock")
print(f"📦 Updated stock (single field write): {updated_stock}")
print()
print("💡 Redis Hashes are great when you need to read/update individual fields.")
print("   JSON strings are simpler and fine for most use cases.")

## 🧹 Cleanup

In [ ]:
r.flushdb()
print("🧹 Redis cleared")

## 📚 Summary

### Cache-Aside Pattern

| Aspect | Detail |
|--------|--------|
| **How** | App checks cache → miss → query DB → store in cache |
| **Pros** | Simple, only caches what's needed, cache stays lean |
| **Cons** | First request is slow (miss), data can go stale |
| **Invalidation** | Delete cache key when DB is updated |
| **Best for** | Most use cases — this is the default pattern |

### Key Takeaways

1. **Cache-aside is the default** — use it unless you have a specific reason not to
2. **Lazy loading** means only requested data gets cached
3. **Staleness is the main risk** — always invalidate on writes
4. **Redis Hashes** are useful when you need per-field access

### Next Up

In **Notebook 3**, we'll explore **Write-Through** and **Write-Behind** patterns — where the cache is kept in sync during writes, not just reads.